In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/0b/0nt_hmld68j1x9vbn6980snc0000gn/T/ipykernel_4110/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [9]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


In [11]:
data.head(1000)

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [13]:
from sklearn.model_selection import train_test_split

# Features and target
X = data['text']
y = data['label']

# Train-test split (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print("\nClass distribution in training set:\n", y_train.value_counts(normalize=True))
print("\nClass distribution in test set:\n", y_test.value_counts(normalize=True))

Train set size: 800
Test set size: 200

Class distribution in training set:
 label
0    0.5575
1    0.4425
Name: proportion, dtype: float64

Class distribution in test set:
 label
0    0.56
1    0.44
Name: proportion, dtype: float64


## Data Preprocessing

In [15]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


LookupError: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - '/Users/iciarroldanares/nltk_data'
    - '/opt/anaconda3/nltk_data'
    - '/opt/anaconda3/share/nltk_data'
    - '/opt/anaconda3/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [19]:
import re
import html

# 1) Remove <script> and <style> blocks (inline JS/CSS)
SCRIPT_RE = re.compile(r"<script\b[^>]*>.*?</script\s*>", flags=re.IGNORECASE | re.DOTALL)
STYLE_RE  = re.compile(r"<style\b[^>]*>.*?</style\s*>",  flags=re.IGNORECASE | re.DOTALL)

# 2) Remove HTML comments (must be before generic tag removal)
COMMENT_RE = re.compile(r"<!--.*?-->", flags=re.DOTALL)

# 3) Remove any remaining tags like <div ...>...</div> or self-closing <br/>
TAG_RE = re.compile(r"<[^>]+>")

# 4) Collapse whitespace
MULTISPACE_RE = re.compile(r"\s+")

def strip_html(text: str) -> str:
    if not isinstance(text, str):
        text = "" if text is None else str(text)

    # Order matters:
    text = SCRIPT_RE.sub(" ", text)     # inline JS
    text = STYLE_RE.sub(" ", text)      # inline CSS
    text = COMMENT_RE.sub(" ", text)    # html comments
    text = TAG_RE.sub(" ", text)        # remaining tags

    # Decode HTML entities, e.g., &amp; -> &, &nbsp; -> space
    text = html.unescape(text)

    # Normalize whitespace
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

# Apply to your dataframe
data["text_nohtml"] = data["text"].apply(strip_html)
data[["text", "text_nohtml"]].head()


,text,text_nohtml
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...","DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL..."
1,Will do.,Will do.
2,Nora--Cheryl has emailed dozens of memos about...,Nora--Cheryl has emailed dozens of memos about...
3,Dear Sir=2FMadam=2C I know that this proposal ...,Dear Sir=2FMadam=2C I know that this proposal ...
4,fyi,fyi


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [22]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    
    # Remove all special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    
    # Remove single characters (isolated letters)
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)
    
    # Remove single characters from the start of text
    text = re.sub(r"^[a-zA-Z]\s+", " ", text)
    
    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text)
    
    # Remove prefixed 'b' (common when reading bytes objects)
    text = re.sub(r"^b\s+", "", text)
    
    # Convert to lowercase
    text = text.lower().strip()
    
    return text

# Apply to your no-HTML-cleaned column
data["text_clean"] = data["text_nohtml"].apply(clean_text)
data[["text_nohtml", "text_clean"]].head()


,text_nohtml,text_clean
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...
1,Will do.,will do
2,Nora--Cheryl has emailed dozens of memos about...,nora cheryl has emailed dozens of memos about ...
3,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam i know that this proposal migh...
4,fyi,fyi


## Now let's work on removing stopwords
Remove the stopwords.

In [46]:
import nltk
nltk.download("stopwords")
nltk.download("punkt")   # if not already downloaded
nltk.download("wordnet") # if not already downloaded

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/iciarroldanares/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/iciarroldanares/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/iciarroldanares/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [34]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
print("Number of stopwords:", len(stop_words))
print(list(stop_words)[:20])

Number of stopwords: 198
['before', 'we', "we're", 'then', 'should', 'they', 'herself', 'ain', 'than', "haven't", "it's", 'all', 'll', 'over', 'him', 'needn', 'o', 'now', 've', 'will']


In [48]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load English stopwords
stop_words = set(stopwords.words("english"))

def remove_stopwords(text: str) -> str:
    tokens = word_tokenize(text)  # split into words
    filtered = [w for w in tokens if w not in stop_words]  # remove stopwords
    return " ".join(filtered)

# Apply to clean column
data["text_nostop"] = data["text_clean"].apply(remove_stopwords)

# Check results
data[["text_clean", "text_nostop"]].head()


,text_clean,text_nostop
0,dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...
1,will do,
2,nora cheryl has emailed dozens of memos about ...,nora cheryl emailed dozens memos haiti weekend...
3,dear sir fmadam i know that this proposal migh...,dear sir fmadam know proposal might surprise e...
4,fyi,fyi


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [51]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Make sure resources are available
nltk.download("wordnet")
nltk.download("omw-1.4")  

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text: str) -> str:
    tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmas)

# Apply to the stopword-free column
data["text_lemma"] = data["text_nostop"].apply(lemmatize_text)

# Preview results
data[["text_nostop", "text_lemma"]].head()


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/iciarroldanares/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/iciarroldanares/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,text_nostop,text_lemma
0,dear sir strictly private business proposal mi...,dear sir strictly private business proposal mi...
1,,
2,nora cheryl emailed dozens memos haiti weekend...,nora cheryl emailed dozen memo haiti weekend p...
3,dear sir fmadam know proposal might surprise e...,dear sir fmadam know proposal might surprise e...
4,fyi,fyi


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [54]:
from sklearn.feature_extraction.text import CountVectorizer

In [56]:
# Separate ham (0) and spam (1)
ham_texts = data[data["label"] == 0]["text_lemma"]
spam_texts = data[data["label"] == 1]["text_lemma"]

# Bag of Words for ham
vectorizer_ham = CountVectorizer()
X_ham = vectorizer_ham.fit_transform(ham_texts)
ham_counts = pd.DataFrame(
    X_ham.toarray().sum(axis=0),
    index=vectorizer_ham.get_feature_names_out(),
    columns=["count"]
).sort_values(by="count", ascending=False).head(10)

# Bag of Words for spam
vectorizer_spam = CountVectorizer()
X_spam = vectorizer_spam.fit_transform(spam_texts)
spam_counts = pd.DataFrame(
    X_spam.toarray().sum(axis=0),
    index=vectorizer_spam.get_feature_names_out(),
    columns=["count"]
).sort_values(by="count", ascending=False).head(10)

print("Top 10 words in HAM messages:\n", ham_counts, "\n")
print("Top 10 words in SPAM messages:\n", spam_counts)

Top 10 words in HAM messages:
            count
state        136
pm           127
would        107
president     99
time          95
call          94
mr            91
obama         84
percent       81
secretary     79 

Top 10 words in SPAM messages:
              count
money          987
account        899
bank           801
fund           782
transaction    555
business       514
country        513
mr             490
million        463
transfer       426


In [61]:
from sklearn.model_selection import train_test_split

# Split into train/validation (e.g., 80/20)
data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    stratify=data["label"],
    random_state=42
)

print("Train size:", data_train.shape)
print("Validation size:", data_val.shape)


Train size: (800, 6)
Validation size: (200, 6)


## Extra features

In [85]:
import re, html, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

# Make sure required NLTK data exists
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

STOP = set(stopwords.words("english"))
LEMMA = WordNetLemmatizer()

SCRIPT_RE  = re.compile(r"<script\b[^>]*>.*?</script\s*>", flags=re.I|re.S)
STYLE_RE   = re.compile(r"<style\b[^>]*>.*?</style\s*>",  flags=re.I|re.S)
COMMENT_RE = re.compile(r"<!--.*?-->", flags=re.S)
TAG_RE     = re.compile(r"<[^>]+>")
MULTI_WS   = re.compile(r"\s+")

def strip_html(s: str) -> str:
    s = "" if s is None else str(s)
    s = SCRIPT_RE.sub(" ", s)
    s = STYLE_RE.sub(" ", s)
    s = COMMENT_RE.sub(" ", s)
    s = TAG_RE.sub(" ", s)
    s = html.unescape(s)
    return MULTI_WS.sub(" ", s).strip()

def clean_text(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9\s]", " ", s)     # special chars
    s = re.sub(r"\d+", " ", s)                # numbers
    s = re.sub(r"\s+[a-zA-Z]\s+", " ", s)     # single letters
    s = re.sub(r"^[a-zA-Z]\s+", " ", s)       # single at start
    s = re.sub(r"\s+", " ", s)                # multi spaces
    s = re.sub(r"^b\s+", "", s)               # prefixed 'b'
    return s.lower().strip()

def remove_stop(s: str) -> str:
    toks = word_tokenize(s)
    return " ".join([t for t in toks if t not in STOP])

def lemmatize_text(s: str) -> str:
    toks = word_tokenize(s)
    return " ".join(LEMMA.lemmatize(t) for t in toks)

def ensure_preprocessed(df):
    # If a processed column already exists, reuse it
    if 'preprocessed_text' in df.columns:
        return df
    for col in ['text_lemma', 'text_nostop', 'text_clean']:
        if col in df.columns:
            df['preprocessed_text'] = df[col]
            return df
    df['preprocessed_text'] = (
        df['text'].fillna("").map(strip_html).map(clean_text).map(remove_stop).map(lemmatize_text)
    )
    return df

if 'data_train' not in globals() or 'data_val' not in globals():
    data_train, data_val = train_test_split(
        data, test_size=0.2, stratify=data['label'], random_state=42
    )

data_train = ensure_preprocessed(data_train)
data_val   = ensure_preprocessed(data_val)

# Extra features
money_words = ["euro", "dollar", "pound", "eur", "usd", "gbp"]
money_symbols = ["€", "$"]
money_regex = r"(?:\b(?:" + "|".join(map(re.escape, money_words)) + r")\b|" + "|".join(map(re.escape, money_symbols)) + r")"

suspicious_words = [
    "free","cheap","sex","money","account","bank","fund",
    "transfer","transaction","win","deposit","password"
]
susp_regex = r"\b(?:" + "|".join(map(re.escape, suspicious_words)) + r")\b"

for df in (data_train, data_val):
    df['money_mark'] = df['preprocessed_text'].str.contains(money_regex, case=False, regex=True, na=False).astype(int)
    df['suspicious_words'] = df['preprocessed_text'].str.contains(susp_regex, case=False, regex=True, na=False).astype(int)
    df['text_len'] = df['preprocessed_text'].str.len()

data_train.head()

,text,label,text_nohtml,text_clean,text_nostop,text_lemma,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,Dear=2C Good day hope fine=2Cdear am writting ...,dear good day hope fine cdear am writting this...,dear good day hope fine cdear writting mail du...,dear good day hope fine cdear writting mail du...,dear good day hope fine cdear writting mail du...,1,1,1002
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,from mr henry kaborethe chief auditor incharge...,mr henry kaborethe chief auditor inchargeforei...,mr henry kaborethe chief auditor inchargeforei...,mr henry kaborethe chief auditor inchargeforei...,0,1,1946
971,Will do.,0,Will do.,will do,,,,0,0,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND ...,from the desk of dr adamu ismalerauditing and ...,desk dr adamu ismalerauditing accounting manag...,desk dr adamu ismalerauditing accounting manag...,desk dr adamu ismalerauditing accounting manag...,1,1,383
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,"Dear Friend, My name is LOI C.ESTRADA,The wife...",dear friend my name is loi estrada the wife of...,dear friend name loi estrada wife mr josephest...,dear friend name loi estrada wife mr josephest...,dear friend name loi estrada wife mr josephest...,1,1,1475


## How would you create a Bag of Words with the CountVectorizer method?

In [87]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize vectorizer
vectorizer = CountVectorizer()

In [89]:
# Fit on the training text and transform into BoW representation
X_bow = vectorizer.fit_transform(data['preprocessed_text'])

print("Shape of BoW matrix:", X_bow.shape)

Shape of BoW matrix: (1000, 29748)


In [93]:
# Vocabulary 
print(vectorizer.vocabulary_)

# Get feature names
print(vectorizer.get_feature_names_out()[:20])  

{'dear': 5797, 'sir': 22554, 'strictly': 23180, 'private': 19571, 'business': 3681, 'proposal': 19692, 'mike': 15779, 'chukwu': 4393, 'manager': 15302, 'bill': 3183, 'exchange': 8052, 'foreign': 8898, 'remittance': 20915, 'department': 5975, 'zenith': 29184, 'international': 12283, 'bank': 2740, 'plc': 19109, 'writing': 27843, 'letter': 14447, 'ask': 2121, 'support': 23330, 'cooperation': 5207, 'carry': 3998, 'opportunity': 18085, 'discovered': 6343, 'abandoned': 73, 'sum': 23305, 'fifteen': 8595, 'million': 15799, 'united': 25574, 'state': 23052, 'dollar': 6574, 'account': 247, 'belongs': 2990, 'one': 17965, 'customer': 5530, 'died': 6246, 'along': 1214, 'entire': 7612, 'family': 8281, 'wife': 27503, 'two': 25006, 'child': 4347, 'november': 17154, 'plane': 19086, 'crash': 5366, 'since': 22519, 'heard': 10577, 'death': 5803, 'expecting': 8105, 'next': 16792, 'kin': 13713, 'come': 4767, 'put': 19888, 'claim': 4506, 'money': 16033, 'heir': 10615, 'release': 20856, 'fund': 9287, 'unless':

In [101]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

bow_df.head()

,aa,aaa,aabeiawaeaambiqaceqedeqh,aac,aaccount,aacw,aae,aaecaxeebsexbhjbuqdhcrmimoeifekrobhbcsmzuvavynlrchyknoel,aaegmdbsch,aaeh,...,zzwh,zzwqgb,zzwqgdg,zzwqgyw,zzwx,zzxh,zzxmsihdoawxlig,zzz,zzzahbxntxe,zzzj
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TD-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [99]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1) Initialize vectorizer
vectorizer = TfidfVectorizer()

# 2) Fit on the whole dataset and transform into TF-IDF matrix
X_tfidf = vectorizer.fit_transform(data['preprocessed_text'])

# 3) Print the shape
print("Shape of TF-IDF matrix:", X_tfidf.shape)

Shape of TF-IDF matrix: (1000, 29748)


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to find the **best feature representation**.

You can work with teams of two persons (recommended).

In [ ]:
# Your code